# Two Maps · **Silver layer** — and the reconciliation gate

**What this notebook does, in one breath:** turns the Bronze files into tables
with an asserted grain, then refuses to go further until OHID's published
England HLE figure is reproduced from my own extract.

Four Silver tables, one gate:

1. **HLE** — ONS and Fingertips on one schema, grain MSOA × sex.
2. **Gate** — England headline, row-for-row MSOA match, bulletin distribution stats, RTT vs press notice.
3. **Geography** — MSOA → LAD → region → Sub-ICB → ICB, with ICB nesting *asserted*, not assumed.
4. **RTT** — provider grain, 18-week and 52-week shares recomputed from bands.
5. **Catchment** — the MSOA × trust crosswalk, grain asserted.

## Step 0 — Tools

In [1]:
import importlib.util, sys
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
PIPELINE = PROJECT_DIR / "src" / "pipeline"
sys.path.insert(0, str(PIPELINE))
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 120)

def run_step(name: str):
    """Import and run one pipeline step by filename (they start with digits, so importlib)."""
    spec = importlib.util.spec_from_file_location(name, PIPELINE / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.main()
    return mod

BRONZE = PROJECT_DIR / "data" / "bronze"
SILVER = PROJECT_DIR / "data" / "silver"
GOLD = PROJECT_DIR / "data" / "gold"
print(PROJECT_DIR)

/Users/yusufismail/Two_maps


## Step 1 — HLE on one schema

The ONS file has more than MSOAs in it — local authorities, regions, the old
42 ICBs, combined authorities. Silver keeps all of it (the 42-ICB rows become a
useful independent benchmark in Gold) but the grain is `area_code × sex` and
that is asserted. The excluded Barnet 042 male row is carried with an explicit
reason rather than dropped.

In [2]:
run_step('02_silver_hle')

ONS rows: 15356 | area types: {'MSOA': 14528, 'Local Areas': 678, 'Integrated Care Board': 84, 'Combined Authority': 30, 'Region': 18, 'Welsh Health Board': 14, 'Country': 4}
ONS English MSOAs: 6856 | rows with HLE: 13711 | excluded: 1
Fingertips rows: 13714 | area types: {'MSOA': 13712, 'England': 2} | MSOAs: 6856 | null HLE: 1


<module '02_silver_hle' from '/Users/yusufismail/Two_maps/src/pipeline/02_silver_hle.py'>

In [3]:
ons = pd.read_parquet(SILVER / "hle" / "hle_ons.parquet")
ons[ons.hle_excluded_reason != ""]

,source,period,country,area_type,area_code,area_name,sex,hle,hle_lci,hle_uci,pct_life_in_good_health,le,le_lci,le_uci,le_imputed_age_groups,health_prev_imputed,le_implausible,hle_excluded_reason
15355,ONS hslemsoa.xlsx (30 Jul 2026),2019 to 2023,England,MSOA,E02006953,Barnet 042,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,97.1,"LE implausible (ONS outer-fence rule, sheet 4); ONS publishes no HSLE"


## Step 2 — The gate

HLE is a life-table quantity. You cannot rebuild England's 65.1 by averaging
6,856 MSOA values — it is a separate calculation on pooled deaths and
populations. So "reproduce OHID's published figure" has to mean something
checkable:

- **A.** Fingertips England row equals the ONS England row, to the decimal, CIs included.
- **B.** Every English MSOA × sex value in Fingertips equals the ONS value.
- **C.** The ONS bulletin's own distribution statistics — median, IQR, 10th and 90th percentiles — reproduce from my rows.
- **D.** The RTT extract reconciles to NHS England's press notice.

The script exits non-zero on failure; nothing downstream runs.

In [4]:
run_step('03_reconciliation_gate')

                  check    sex                       metric      ons  fingertips  diff  pass                                                                   note
     A_england_headline   Male                          hle     65.1       65.10  0.00  True                                                                    NaN
     A_england_headline   Male                      hle_lci     65.1       65.10  0.00  True                                                                    NaN
     A_england_headline   Male                      hle_uci     65.1       65.10  0.00  True                                                                    NaN
     A_england_headline Female                          hle     66.5       66.50  0.00  True                                                                    NaN
     A_england_headline Female                      hle_lci     66.5       66.50  0.00  True                                                                    NaN
     A_england_h

<module '03_reconciliation_gate' from '/Users/yusufismail/Two_maps/src/pipeline/03_reconciliation_gate.py'>

All pass. Two things I want on the record from getting here:

- My first version of check C tested the bulletin's "57.7 to 71.0" as a min/max.
  It failed, because those are the 10th and 90th percentiles — my transcription
  error, not a data problem. The gate now tests what the bulletin actually says.
- The male interdecile range comes out 13.3 from the published 1 d.p. values
  against 13.4 in the bulletin; ONS computes it from unrounded internals. Tolerated
  at 0.15 with a note in the CSV, not hidden.
- The 65.1 / 66.5 here is **Census-2021-based** HLE. It is not the ~61–62 APS-based
  England HLE people usually quote, and ONS says the two are not comparable.
  The article must not mix them.

## Step 3 — Geography, and the ICB assertion

MSOAs are built from LSOAs; ICBs are built from LSOAs; so an MSOA *should* sit
inside one ICB. I don't assume it — Silver counts distinct ICBs per MSOA and
writes any that straddle to a side table. Two wrinkles turned up:

- The OA-level lookup's LAD best-fit puts one LSOA (North Somerset 005) in two
  LADs. LAD is taken from the LSOA-level April 2026 lookup instead.
- Barnsley and Sheffield were re-coded in the April 2026 LAD vintage and miss the
  Dec 2024 LAD→Region lookup; resolved by name, asserted nothing else falls through.

In [5]:
run_step('04_silver_geography')

LSOA merge: {'both': 33755, 'left_only': 0, 'right_only': 0}
MSOAs straddling >1 ICB: 5 | >1 SICBL: 22 | >1 LAD26: 0
MSOAs with region resolved by LAD name (2026 re-coding): 100 -> ['Barnsley', 'Sheffield']
ICBs: 36 | SICBLs: 106 | LADs: 296 | regions: 9
{'count': 36.0, 'mean': 190.0, 'std': 98.0, 'min': 63.0, '25%': 115.0, '50%': 166.0, '75%': 228.0, 'max': 401.0}


<module '04_silver_geography' from '/Users/yusufismail/Two_maps/src/pipeline/04_silver_geography.py'>

In [6]:
geo = pd.read_parquet(SILVER / "geography" / "msoa_geography.parquet")
display(geo[geo.straddles_icb][["msoa_code", "msoa_name", "icb_name_short", "n_lsoa", "n_icb"]])
geo.groupby("icb_name_short").size().sort_values().rename("n_msoa").to_frame().T

,msoa_code,msoa_name,icb_name_short,n_lsoa,n_icb
3496,E02003654,Buckinghamshire 003,Thames Valley,4,2
3499,E02003657,Buckinghamshire 006,Central East,4,2
3841,E02004007,Copeland 008,Lancashire and South Cumbria,7,2
5514,E02005742,Craven 001,Lancashire and South Cumbria,4,2
6632,E02006886,Vale of White Horse 016,"Bath and North East Somerset, Swindon and Wiltshire",7,2


icb_name_short,"Shropshire, Telford and Wrekin",Somerset,Cornwall and the Isles of Scilly,Gloucestershire,Lincolnshire,Northamptonshire,Dorset,Herefordshire and Worcestershire,Coventry and Warwickshire,"Bristol, North Somerset and South Gloucestershire","Bath and North East Somerset, Swindon and Wiltshire","Leicester, Leicestershire and Rutland",Derby and Derbyshire,Nottingham and Nottinghamshire,Staffordshire and Stoke-on-Trent,Black Country,Devon,Birmingham and Solihull,South Yorkshire,South West London,Norfolk and Suffolk,South East London,Essex,Humber and North Yorkshire,Lancashire and South Cumbria,North East London,Kent and Medway,Hampshire and Isle of Wight,Thames Valley,West Yorkshire,Cheshire and Merseyside,Greater Manchester,Surrey and Sussex,Central East,North East and North Cumbria,West and North London
n_msoa,63,71,74,76,88,93,96,108,109,117,119,127,130,137,141,154,158,161,171,175,203,208,215,215,216,218,221,247,261,307,325,353,353,364,381,401


Five MSOAs straddle two ICBs (assigned by LSOA majority, flagged); 22 straddle
two Sub-ICB locations. ICB sizes run from 63 MSOAs (Somerset) to 401 (Cheshire
and Merseyside).

## Step 4 — RTT at provider grain

The extract is provider × commissioner × treatment function × 105 weekly bands.
Silver rolls commissioner out, rebuilds the total from the bands (the `Total`
column is blank for incomplete pathways), and recomputes the 18-week and 52-week
shares rather than trusting any published percentage. The `C_999` total row is
asserted equal to the sum of the named treatment functions for every provider.

In [7]:
run_step('05_silver_rtt')

raw rows: 182372 | period: ['RTT-July-2026']
providers: 533 | kinds: {'independent sector (N-code)': 217, 'other (non-R/N ODS code)': 165, 'NHS trust': 151}
England incomplete pathways: 7,162,692 | within 18wk: 65.3% | >52wk: 111,429 (1.56%)
ICB parents: 36


<module '05_silver_rtt' from '/Users/yusufismail/Two_maps/src/pipeline/05_silver_rtt.py'>

In [8]:
prov = pd.read_parquet(SILVER / "rtt" / "rtt_provider.parquet")
prov.sort_values("total_waiting", ascending=False).head(10)[["provider_code", "provider_name", "provider_icb_name", "total_waiting", "pct_within_18", "pct_over_52"]]

,provider_code,provider_name,provider_icb_name,total_waiting,pct_within_18,pct_over_52
348,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,NHS GREATER MANCHESTER INTEGRATED CARE BOARD,167649.0,0.638882,0.008291
365,RAJ,MID AND SOUTH ESSEX NHS FOUNDATION TRUST,NHS ESSEX INTEGRATED CARE BOARD,165420.0,0.497763,0.079217
366,RAL,ROYAL FREE LONDON NHS FOUNDATION TRUST,NHS WEST AND NORTH LONDON INTEGRATED CARE BOARD,139727.0,0.627223,0.011079
427,RM3,NORTHERN CARE ALLIANCE NHS FOUNDATION TRUST,NHS GREATER MANCHESTER INTEGRATED CARE BOARD,134561.0,0.558624,0.028069
356,R1H,BARTS HEALTH NHS TRUST,NHS NORTH EAST LONDON INTEGRATED CARE BOARD,121618.0,0.637743,0.008469
496,RYR,UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST,NHS SURREY AND SUSSEX INTEGRATED CARE BOARD,117217.0,0.530827,0.043688
474,RWE,UNIVERSITY HOSPITALS OF LEICESTER NHS TRUST,"NHS LEICESTER, LEICESTERSHIRE AND RUTLAND INTEGRATED CARE BOARD",116558.0,0.600053,0.020651
408,RJ1,GUY'S AND ST THOMAS' NHS FOUNDATION TRUST,NHS SOUTH EAST LONDON INTEGRATED CARE BOARD,109056.0,0.682420,0.006630
461,RTG,UNIVERSITY HOSPITALS OF DERBY AND BURTON NHS FOUNDATION TRUST,NHS DERBY AND DERBYSHIRE INTEGRATED CARE BOARD,99108.0,0.607186,0.020574
386,RDE,EAST SUFFOLK AND NORTH ESSEX NHS FOUNDATION TRUST,NHS ESSEX INTEGRATED CARE BOARD,95757.0,0.596687,0.005190


533 providers report incomplete pathways; only 151 are NHS trusts. That ratio
matters for the crosswalk: the catchment file covers 134 *acute* trusts and
nothing else.

## Step 5 — The crosswalk, as published

Three tables (all / elective / emergency admissions), two catchment years. RTT
is an elective measure, so Gold joins on the elective catchment. Rounding and
suppression are kept exactly as OHID published them.

In [9]:
run_step('06_silver_catchment')

rows: 285248 | years: [2023, 2024] | trusts: 134 | MSOAs: 6856
2024 all: MSOAs with exactly one FPTP trust: 6856 of 6856 | FPTP share quantiles: {0.1: 0.543, 0.5: 0.792, 0.9: 0.918} | trusts per MSOA median: 8
2024 elective: MSOAs with exactly one FPTP trust: 6856 of 6856 | FPTP share quantiles: {0.1: 0.522, 0.5: 0.758, 0.9: 0.912} | trusts per MSOA median: 7
2024 emergency: MSOAs with exactly one FPTP trust: 6856 of 6856 | FPTP share quantiles: {0.1: 0.591, 0.5: 0.86, 0.9: 0.944} | trusts per MSOA median: 5
trust types: {'Acute - Teaching': 49, 'Acute - Large': 23, 'Acute - Medium': 20, 'Acute - Small': 19, 'Acute - Specialist': 16, 'Acute - Multi-service': 7}


<module '06_silver_catchment' from '/Users/yusufismail/Two_maps/src/pipeline/06_silver_catchment.py'>

In [10]:
cat = pd.read_parquet(SILVER / "catchment" / "msoa_trust_admissions.parquet")
ex = cat[(cat.catchment_year == 2024) & (cat.admission_type == "elective") & (cat.msoa_code == "E02003654")]  # Buckinghamshire 003, one of the ICB-straddlers
ex.sort_values("patients_admitted", ascending=False)[["trust_name", "patients_admitted", "share_of_msoa_patients", "fptp", "share_band"]]

,trust_name,patients_admitted,share_of_msoa_patients,fptp,share_band
178954,Milton Keynes University Hospital NHS Foundation Trust,665,0.569,True,40% to 59%
215416,Buckinghamshire Healthcare NHS Trust,245,0.208,False,20% to 39%
208317,Oxford University Hospitals NHS Foundation Trust,120,0.103,False,05% to 19%
177580,Bedfordshire Hospitals NHS Foundation Trust,30,0.027,False,Less than 5% not shown
186661,Guy's and St Thomas' NHS Foundation Trust,20,0.019,False,Less than 5% not shown
195172,Northampton General Hospital NHS Trust,10,0.007,False,Less than 5% not shown
216776,Imperial College Healthcare NHS Trust,10,0.008,False,Less than 5% not shown


One neighbourhood, seven trusts with ≥8 elective admissions, the dominant one
taking a bit over half. That is what "a crosswalk exists but isn't clean"
looks like at the row level. Gold measures how typical this is.